# Model Training
Train a virality regressor and persist artifacts.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

df = pd.read_csv(Path('..') / 'data' / 'Instagram_Analytics.csv')
for col in ['likes', 'comments', 'shares', 'saves', 'reach', 'impressions', 'caption_length', 'hashtags_count']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df['viral_score'] = (df['likes'].rank(pct=True) * 40 + df['shares'].rank(pct=True) * 30 + df['comments'].rank(pct=True) * 30)
X = df[['likes', 'comments', 'shares', 'saves', 'reach', 'impressions', 'caption_length', 'hashtags_count']]
y = df['viral_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print('MAE', round(mean_absolute_error(y_test, pred), 3))
print('R2', round(r2_score(y_test, pred), 3))

save_dir = Path('..') / 'backend' / 'saved_models'
save_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, save_dir / 'viral_model_notebook.pkl')